# 00 — Semis Intraday Data Download

Strategy context: **Cross-Sectional Equity Dispersion / Basket StatArb**

Purpose:
- Define the semiconductor starter universe used by the research notebook.
- Download required 1-minute OHLCV bars from Polygon/Massive into ArcticDB.
- Audit data coverage for regular US session minutes.
- Repair partial downloads and historical gaps in small, rate-limit-friendly windows.

This notebook is data plumbing only. It should not compute signals, portfolios, PnL, or research decisions.


## 1. Operating Rules

- Keep network calls disabled by default.
- Download one or a few tickers at a time.
- Prefer monthly or weekly windows if Polygon returns `429`.
- Restart the kernel after a download batch before loading/auditing with a fresh ArcticDB handle if LMDB warns about an already-open path.
- Treat `audit_gaps` output as approximate because the first version uses business days, not a full NYSE holiday/early-close calendar.


## 2. Imports and Repo Setup


In [1]:
import sys
from pathlib import Path
from pprint import pprint

import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
        if (candidate / "Bluegrey" / "pyproject.toml").exists():
            return candidate / "Bluegrey"
    raise RuntimeError("Could not find Bluegrey repo root.")

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

NOTEBOOK_DIR = REPO_ROOT / "research" / "Jordi" / "cross_sectional_equity_dispersion"
REPORT_DIR = NOTEBOOK_DIR / "outputs" / "data_download"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Repo root:", REPO_ROOT)
print("Report dir:", REPORT_DIR)


Repo root: /Users/marina/Documents/jordi/projects/Trading/bluegrey/1.Cloud/Bluegrey
Report dir: /Users/marina/Documents/jordi/projects/Trading/bluegrey/1.Cloud/Bluegrey/research/Jordi/cross_sectional_equity_dispersion/outputs/data_download


## 3. Download Configuration

Stocks Starter has a five-year historical window. Keep `REQUEST_START_DATE` inside that window and use smaller windows while bootstrapping.


In [2]:
from src.config import config as bg_config

CONFIG = {
    "strategy_name": "cross_sectional_equity_dispersion",
    "sector": "semiconductors",
    "primary_etf": "SMH",
    "alternative_etf": "SOXX",
    "timezone": "America/New_York",
    "regular_session_start": "09:30",
    "regular_session_end": "16:00",
    "bar_size": "1min",
    "arctic_library": "equity_min",
    "vendor": "polygon_massive",
}

REQUEST_START_DATE = "2024-01-01"
REQUEST_END_DATE = "2026-01-31"

REQUEST_START_DATE = "2024-06-20"
REQUEST_END_DATE = "2026-06-01"

REQUEST_START_DATE = "2021-07-01"
REQUEST_END_DATE = "2026-06-12"

#DOWNLOAD_MODE = "repair_gaps"
DOWNLOAD_MODE = "force_window"


# Valid modes:
# - "audit_gaps": no network, inspect missing regular-session bars.
# - "update_forward": smart update from last timestamp minus one day.
# - "force_window": download exactly REQUEST_START_DATE -> REQUEST_END_DATE.
# - "repair_gaps": audit gaps and repair missing windows.

RUN_DOWNLOAD = True
SYMBOL_LIMIT = 30
ONLY_MISSING_SYMBOLS = False

MAX_WINDOW_DAYS = 3
MAX_WINDOWS_PER_TICKER = 30
SLEEP_SECONDS = 3
MAX_RETRIES = 1
RETRY_SLEEP_SECONDS = 300.0

print("Arctic path:", bg_config.ARCTIC_PATH)
pprint(CONFIG)


Arctic path: lmdb:///Users/marina/Documents/jordi/projects/Trading/bluegrey/1.Cloud/Bluegrey/src/data/arctic_db?map_size=100GB
{'alternative_etf': 'SOXX',
 'arctic_library': 'equity_min',
 'bar_size': '1min',
 'primary_etf': 'SMH',
 'regular_session_end': '16:00',
 'regular_session_start': '09:30',
 'sector': 'semiconductors',
 'strategy_name': 'cross_sectional_equity_dispersion',
 'timezone': 'America/New_York',
 'vendor': 'polygon_massive'}


## 4. Static Semiconductor Universe

This mirrors the starter universe in `01_cross_sectional_dispersion_semis_research.ipynb`.


In [3]:
SEMIS_STARTER_UNIVERSE = [
    "NVDA", "AMD", "AVGO", "INTC", "QCOM", "TXN", "MU", "MRVL",
    "AMAT", "LRCX", "KLAC", "MCHP", "ON", "ADI", "NXPI", "MPWR",
    "SWKS", "QRVO", "TER", "WDC", "STX", "COHR", "LSCC", "CRUS",
    "ACLS", "AEIS", "FORM",
]

ETF = CONFIG["primary_etf"]
ALT_ETF = CONFIG["alternative_etf"]

STOCK_SYMBOLS = sorted(set(SEMIS_STARTER_UNIVERSE))
DATA_SYMBOLS = sorted(set(STOCK_SYMBOLS + [ETF, ALT_ETF]))

universe = pd.DataFrame({
    "symbol": DATA_SYMBOLS,
    "role": ["primary_etf" if s == ETF else "alternative_etf" if s == ALT_ETF else "stock" for s in DATA_SYMBOLS],
    "sector": CONFIG["sector"],
})

print("Stocks:", len(STOCK_SYMBOLS))
print("Data symbols including ETFs:", len(DATA_SYMBOLS))
display(universe)


Stocks: 27
Data symbols including ETFs: 29


,symbol,role,sector
0,ACLS,stock,semiconductors
1,ADI,stock,semiconductors
2,AEIS,stock,semiconductors
3,AMAT,stock,semiconductors
4,AMD,stock,semiconductors
5,AVGO,stock,semiconductors
6,COHR,stock,semiconductors
7,CRUS,stock,semiconductors
8,FORM,stock,semiconductors
9,INTC,stock,semiconductors


## 5. ArcticDB Availability

This only checks which symbols exist in the library. It does not prove the full range is complete.


In [ ]:
import arcticdb as adb
from src.config import config as bg_config

arctic = adb.Arctic(bg_config.ARCTIC_PATH)
library_name = CONFIG["arctic_library"]

if library_name not in arctic.list_libraries():
    arctic.create_library(library_name)

lib = arctic[library_name]
available_symbols = set(lib.list_symbols())

availability = pd.DataFrame({
    "symbol": DATA_SYMBOLS,
    "role": ["primary_etf" if s == ETF else "alternative_etf" if s == ALT_ETF else "stock" for s in DATA_SYMBOLS],
    "exists_in_arctic": [s in available_symbols for s in DATA_SYMBOLS],
})

present_symbols = availability.loc[availability["exists_in_arctic"], "symbol"].tolist()
missing_symbols = availability.loc[~availability["exists_in_arctic"], "symbol"].tolist()

print(f"Library: {library_name}")
print(f"Present: {len(present_symbols)} / {len(DATA_SYMBOLS)}")
print("Missing symbols:", missing_symbols)
display(availability)


## 6. Select Symbols for This Run

Use `SYMBOLS_OVERRIDE` for surgical runs. Keep it small while fighting `429`.


In [5]:
#SYMBOLS_OVERRIDE = ["AMD"]
SYMBOLS_OVERRIDE = None

#SYMBOLS_OVERRIDE = ['INTC', 'KLAC', 'LRCX', 'LSCC', 'MCHP']

if SYMBOLS_OVERRIDE:
    candidate_symbols = [s for s in SYMBOLS_OVERRIDE if s in DATA_SYMBOLS]
elif ONLY_MISSING_SYMBOLS:
    candidate_symbols = missing_symbols
else:
    candidate_symbols = list(DATA_SYMBOLS)

if SYMBOL_LIMIT:
    selected_symbols = candidate_symbols[:SYMBOL_LIMIT]
else:
    selected_symbols = candidate_symbols

print(f"Selected symbols: {len(selected_symbols)}")
print(selected_symbols)


Selected symbols: 29
['ACLS', 'ADI', 'AEIS', 'AMAT', 'AMD', 'AVGO', 'COHR', 'CRUS', 'FORM', 'INTC', 'KLAC', 'LRCX', 'LSCC', 'MCHP', 'MPWR', 'MRVL', 'MU', 'NVDA', 'NXPI', 'ON', 'QCOM', 'QRVO', 'SMH', 'SOXX', 'STX', 'SWKS', 'TER', 'TXN', 'WDC']


## 7. Audit Coverage

This is the safest first step. It makes no Polygon request.


In [6]:
from tools.download_history_polygon import PolygonIngestor

ingestor = PolygonIngestor()

audit_df = ingestor.run_audit_job(
    universe_name="manual_semis_starter",
    start_date=REQUEST_START_DATE,
    end_date=REQUEST_END_DATE,
    lib_name=CONFIG["arctic_library"],
    specific_tickers=selected_symbols,
    timezone=CONFIG["timezone"],
    session_start=CONFIG["regular_session_start"],
    session_end=CONFIG["regular_session_end"],
    max_window_days=MAX_WINDOW_DAYS,
)

audit_path = REPORT_DIR / f"coverage_audit_{REQUEST_START_DATE}_{REQUEST_END_DATE}.csv"
audit_df.to_csv(audit_path, index=False)
print("Saved:", audit_path)
display(audit_df)


20260614 21:29:02.359094 445924 W arcticdb | LMDB path at /Users/marina/Documents/jordi/projects/Trading/bluegrey/1.Cloud/Bluegrey/src/data/arctic_db/ has already been opened in this process which is not supported by LMDB. You should only open a single Arctic instance over a given LMDB path. To continue safely, you should delete this Arctic instance and any others over the LMDB path in this process and then try again. Current process ID=[9063]


   ticker  expected_bars  existing_bars  missing_bars  coverage  missing_windows      first_existing       last_existing
0    ACLS         484710         143072        341638  0.295170              517 2024-06-13 11:04:00 2026-06-09 22:51:00
1     ADI         484710         191140        293570  0.394339              487 2024-06-13 11:23:00 2026-06-09 23:55:00
2    AEIS         484710         102159        382551  0.210763              516 2024-06-13 13:30:00 2026-06-09 22:51:00
3    AMAT         484710         192979        291731  0.398133              353 2024-06-13 08:30:00 2026-06-09 23:59:00
4     AMD         484710         192750        291960  0.397660              310 2024-06-14 08:00:00 2026-06-09 23:59:00
5    AVGO         484710         189234        295476  0.390407              315 2024-06-20 08:05:00 2026-06-01 23:59:00
6    COHR         484710         191501        293209  0.395084              451 2024-06-13 11:54:00 2026-06-09 23:59:00
7    CRUS         484710        

,ticker,start_date,end_date,expected_bars,existing_bars,missing_bars,coverage,missing_windows,first_existing,last_existing
0,ACLS,2021-07-01,2026-06-12,484710,143072,341638,0.295170,517,2024-06-13 11:04:00,2026-06-09 22:51:00
1,ADI,2021-07-01,2026-06-12,484710,191140,293570,0.394339,487,2024-06-13 11:23:00,2026-06-09 23:55:00
2,AEIS,2021-07-01,2026-06-12,484710,102159,382551,0.210763,516,2024-06-13 13:30:00,2026-06-09 22:51:00
3,AMAT,2021-07-01,2026-06-12,484710,192979,291731,0.398133,353,2024-06-13 08:30:00,2026-06-09 23:59:00
4,AMD,2021-07-01,2026-06-12,484710,192750,291960,0.397660,310,2024-06-14 08:00:00,2026-06-09 23:59:00
5,AVGO,2021-07-01,2026-06-12,484710,189234,295476,0.390407,315,2024-06-20 08:05:00,2026-06-01 23:59:00
6,COHR,2021-07-01,2026-06-12,484710,191501,293209,0.395084,451,2024-06-13 11:54:00,2026-06-09 23:59:00
7,CRUS,2021-07-01,2026-06-12,484710,139918,344792,0.288663,517,2024-06-13 12:08:00,2026-06-09 20:00:00
8,FORM,2021-07-01,2026-06-12,484710,154030,330680,0.317778,516,2024-06-13 13:30:00,2026-06-09 23:04:00
9,INTC,2021-07-01,2026-06-12,484710,189236,295474,0.390411,315,2024-06-20 08:00:00,2026-06-01 23:59:00


In [ ]:
report = ingestor.audit_ticker_coverage(
    ticker="AMD",
    lib_name=CONFIG["arctic_library"],
    start_date=REQUEST_START_DATE,
    end_date=REQUEST_END_DATE,
    timezone=CONFIG["timezone"],
    session_start=CONFIG["regular_session_start"],
    session_end=CONFIG["regular_session_end"],
    max_window_days=MAX_WINDOW_DAYS,
)

report["missing_windows"]

## 8. Download or Repair

This cell can call Polygon. It only runs when `RUN_DOWNLOAD = True`.


In [7]:
if not RUN_DOWNLOAD:
    print("RUN_DOWNLOAD is False. No Polygon request was made.")
elif DOWNLOAD_MODE == "audit_gaps":
    print("DOWNLOAD_MODE is audit_gaps. Use the audit cell above; no download requested here.")
elif DOWNLOAD_MODE == "repair_gaps":
    repair_df = ingestor.run_gap_repair_job(
        universe_name="manual_semis_starter",
        timespan="minute",
        multiplier=1,
        start_date=REQUEST_START_DATE,
        end_date=REQUEST_END_DATE,
        lib_name=CONFIG["arctic_library"],
        specific_tickers=selected_symbols,
        timezone=CONFIG["timezone"],
        session_start=CONFIG["regular_session_start"],
        session_end=CONFIG["regular_session_end"],
        max_window_days=MAX_WINDOW_DAYS,
        max_windows_per_ticker=MAX_WINDOWS_PER_TICKER,
        sleep_seconds=SLEEP_SECONDS,
        max_retries=MAX_RETRIES,
        retry_sleep_seconds=RETRY_SLEEP_SECONDS,
    )
    repair_path = REPORT_DIR / f"gap_repair_{REQUEST_START_DATE}_{REQUEST_END_DATE}.csv"
    repair_df.to_csv(repair_path, index=False)
    print("Saved:", repair_path)
    display(repair_df)
elif DOWNLOAD_MODE in {"update_forward", "force_window"}:
    ingestor.run_batch_job(
        universe_name="manual_semis_starter",
        timespan="minute",
        multiplier=1,
        start_date=REQUEST_START_DATE,
        end_date=REQUEST_END_DATE,
        lib_name=CONFIG["arctic_library"],
        specific_tickers=selected_symbols,
        sleep_seconds=SLEEP_SECONDS,
        max_retries=MAX_RETRIES,
        retry_sleep_seconds=RETRY_SLEEP_SECONDS,
        mode=DOWNLOAD_MODE,
    )
else:
    raise ValueError(f"Unknown DOWNLOAD_MODE: {DOWNLOAD_MODE}")


2026-06-14 21:30:50,824 [INFO] --- STARTING BATCH JOB: Universe 'manual_semis_starter' | 1-minute ---
2026-06-14 21:30:50,826 [INFO] Override active: Using manually provided specific_tickers.
2026-06-14 21:30:50,826 [INFO] Data Factory received 29 validated assets. Commencing download...
[1/29] ✅ ACLS: Appended 360154 rows (2021-07-01 -> 2026-06-12).
[2/29] ✅ ADI: Appended 497755 rows (2021-07-01 -> 2026-06-12).
[3/29] ✅ AEIS: Appended 236135 rows (2021-07-01 -> 2026-06-12).
[4/29] ✅ AMAT: Appended 555123 rows (2021-07-01 -> 2026-06-12).
[5/29] ✅ AMD: Appended 958962 rows (2021-07-01 -> 2026-06-12).
[6/29] ✅ AVGO: Appended 650244 rows (2021-07-01 -> 2026-06-12).
[7/29] ✅ COHR: Appended 438482 rows (2021-07-01 -> 2026-06-12).
[8/29] ✅ CRUS: Appended 361021 rows (2021-07-01 -> 2026-06-12).
[9/29] ✅ FORM: Appended 360154 rows (2021-07-01 -> 2026-06-12).
[10/29] ✅ INTC: Appended 892317 rows (2021-07-01 -> 2026-06-12).
[11/29] ✅ KLAC: Appended 442329 rows (2021-07-01 -> 2026-06-12).
[12/29]

In [ ]:
['MPWR','NXPI','MU', 'NVDA', 'NXPI']
['ON', 'QCOM', 'QRVO', 'SMH', 'SOXX']

## 9. Post-Download Audit

After a download/repair batch, restart the kernel if ArcticDB emits the LMDB already-open warning, then rerun setup and this cell.


In [ ]:
from tools.download_history_polygon import PolygonIngestor

if "ingestor" not in globals():
    ingestor = PolygonIngestor()

post_audit_df = ingestor.run_audit_job(
    universe_name="manual_semis_starter",
    start_date=REQUEST_START_DATE,
    end_date=REQUEST_END_DATE,
    lib_name=CONFIG["arctic_library"],
    specific_tickers=selected_symbols,
    timezone=CONFIG["timezone"],
    session_start=CONFIG["regular_session_start"],
    session_end=CONFIG["regular_session_end"],
    max_window_days=MAX_WINDOW_DAYS,
)

post_audit_path = REPORT_DIR / f"coverage_post_audit_{REQUEST_START_DATE}_{REQUEST_END_DATE}.csv"
post_audit_df.to_csv(post_audit_path, index=False)
print("Saved:", post_audit_path)
display(post_audit_df)


In [ ]:
['INTC', 'KLAC', 'LRCX' ,'LSCC', 'MCHP']

## 10. Quick Load Smoke Test

This verifies that `DataStore` can assemble aligned matrices for the selected symbols.


In [ ]:
from src.data.store import DataStore

store = DataStore(CONFIG["arctic_library"])
symbols_to_load = selected_symbols if "selected_symbols" in globals() else DATA_SYMBOLS
data = store.load(symbols_to_load, start_date=REQUEST_START_DATE, end_date=REQUEST_END_DATE)

if not data:
    print("No data loaded for selected symbols.")
else:
    print("Loaded fields:", list(data.keys()))
    print("Close shape:", data["close"].shape)
    print("Range:", data["close"].index.min(), "->", data["close"].index.max())
    display(data["close"].tail())


## 11. Quick Check of needed timeframes

This verifies that theres no gaps between 9:30 and 10am

In [13]:
import pandas as pd
import numpy as np
from src.data.store import DataStore

def fast_window_coverage(
    symbols,
    lib_name="equity_min",
    start_date="2021-07-01",
    end_date="2026-06-12",
    timezone="America/New_York",
    window_start="09:30",
    window_end="10:00",
):
    store = DataStore(lib_name)
    data = store.load(symbols, start_date=start_date, end_date=end_date)

    if not data:
        raise RuntimeError("No data loaded from ArcticDB.")

    close_px = data["close"].copy()

    if close_px.index.tz is None:
        close_px.index = close_px.index.tz_localize("UTC")
    close_px.index = close_px.index.tz_convert(timezone)

    window_px = close_px.between_time(window_start, window_end, inclusive="both")

    expected_bars = (
        int(
            (
                pd.Timestamp(window_end) - pd.Timestamp(window_start)
            ).total_seconds()
            // 60
        )
        + 1
    )

    # Count non-null bars per day x symbol.
    daily_counts = (
        window_px
        .notna()
        .groupby(window_px.index.normalize())
        .sum()
        .astype(int)
    )

    daily_coverage = (daily_counts / expected_bars).clip(upper=1.0)

    has_start = (
        window_px[window_px.index.strftime("%H:%M") == window_start]
        .notna()
        .groupby(window_px[window_px.index.strftime("%H:%M") == window_start].index.normalize())
        .max()
        .reindex(daily_counts.index)
        .fillna(False)
    )

    has_end = (
        window_px[window_px.index.strftime("%H:%M") == window_end]
        .notna()
        .groupby(window_px[window_px.index.strftime("%H:%M") == window_end].index.normalize())
        .max()
        .reindex(daily_counts.index)
        .fillna(False)
    )

    summary = pd.DataFrame({
        "symbol": daily_counts.columns,
        "days_observed": daily_counts.notna().sum().values,
        "avg_window_coverage": daily_coverage.mean().values,
        "full_window_days": (daily_coverage >= 1.0).sum().values,
        "full_window_ratio": (daily_coverage >= 1.0).mean().values,
        "has_start_ratio": has_start.mean().values,
        "has_end_ratio": has_end.mean().values,
        "min_bars_present": daily_counts.min().values,
        "median_bars_present": daily_counts.median().values,
        "avg_bars_present": daily_counts.mean().values,
    }).sort_values("avg_window_coverage", ascending=False)

    return summary, daily_counts, daily_coverage

summary_0930_1000, counts_0930_1000, coverage_0930_1000 = fast_window_coverage(
    symbols=DATA_SYMBOLS,
    lib_name=CONFIG["arctic_library"],
    start_date=REQUEST_START_DATE,
    end_date=REQUEST_END_DATE,
    timezone=CONFIG["timezone"],
    window_start="09:30",
    window_end="10:00",
)

display(summary_0930_1000)

2026-06-15 22:09:55,126 [INFO] 🗄️ Connected to ArcticDB at: lmdb:///Users/marina/Documents/jordi/projects/Trading/bluegrey/1.Cloud/Bluegrey/src/data/arctic_db?map_size=100GB
2026-06-15 22:09:55,129 [INFO] 🔄 Assembling universe matrices for 29 asset(s)...


20260615 22:09:55.126370 445924 W arcticdb | LMDB path at /Users/marina/Documents/jordi/projects/Trading/bluegrey/1.Cloud/Bluegrey/src/data/arctic_db/ has already been opened in this process which is not supported by LMDB. You should only open a single Arctic instance over a given LMDB path. To continue safely, you should delete this Arctic instance and any others over the LMDB path in this process and then try again. Current process ID=[9063]


2026-06-15 22:10:13,350 [INFO] ✅ Data aligned. Master Matrix shape: (1138790, 29)


,symbol,days_observed,avg_window_coverage,full_window_days,full_window_ratio,has_start_ratio,has_end_ratio,min_bars_present,median_bars_present,avg_bars_present
0,ACLS,1242,1.0,1242,1.0,1.0,1.0,31,31.0,31.0
1,ADI,1242,1.0,1242,1.0,1.0,1.0,31,31.0,31.0
2,AEIS,1242,1.0,1242,1.0,1.0,1.0,31,31.0,31.0
3,AMAT,1242,1.0,1242,1.0,1.0,1.0,31,31.0,31.0
4,AMD,1242,1.0,1242,1.0,1.0,1.0,31,31.0,31.0
5,AVGO,1242,1.0,1242,1.0,1.0,1.0,31,31.0,31.0
6,COHR,1242,1.0,1242,1.0,1.0,1.0,31,31.0,31.0
7,CRUS,1242,1.0,1242,1.0,1.0,1.0,31,31.0,31.0
8,FORM,1242,1.0,1242,1.0,1.0,1.0,31,31.0,31.0
9,INTC,1242,1.0,1242,1.0,1.0,1.0,31,31.0,31.0


In [15]:
good_symbols = summary_0930_1000.query(
    "full_window_ratio >= 0.95 and has_start_ratio >= 0.95 and has_end_ratio >= 0.95"
)["symbol"].tolist()

good_symbols

['ACLS',
 'ADI',
 'AEIS',
 'AMAT',
 'AMD',
 'AVGO',
 'COHR',
 'CRUS',
 'FORM',
 'INTC',
 'KLAC',
 'LRCX',
 'LSCC',
 'MCHP',
 'MPWR',
 'MRVL',
 'MU',
 'NVDA',
 'NXPI',
 'ON',
 'QCOM',
 'QRVO',
 'SMH',
 'SOXX',
 'STX',
 'SWKS',
 'TER',
 'TXN',
 'WDC']